# 14 — Unit Testing, Fixtures, and Mocking

Use the numbered concept checks where you need practice. For the engineering route, complete **Write an integration test with a fake clock** and the independent build.

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/14', {1: '86177112244dbdbd', 2: '8d16006c6d5bf5bd', 3: '1730e476028f9349', 4: '788c6f5c495f1e58', 5: '8368b3042b9d137d', 6: '8de54c7c51f0c434', 7: '22c4fa2efa1075a1', 8: '7c5de932e83d5c5e', 9: '3c4500e19f32a46c'}, independent='test_fee')

## 1. unittest basics

- A test class subclasses `unittest.TestCase`; each method starting with `test_` is one test.
- Assert with `self.assertEqual(a, b)`, `assertTrue(x)`, `assertIn(item, items)`, and `with self.assertRaises(ValueError):`.
- In a notebook, run tests with a loader and runner instead of `unittest.main()`.

In [ ]:
import unittest

def add(a, b):
    return a + b

class AddTests(unittest.TestCase):
    def test_numbers(self):
        self.assertEqual(add(2, 3), 5)
    def test_bad_type(self):
        with self.assertRaises(TypeError):
            add(1, "x")

suite = unittest.defaultTestLoader.loadTestsFromTestCase(AddTests)
result = unittest.TextTestRunner(verbosity=0).run(suite)
print(result.testsRun, "tests,", len(result.failures), "failures")

**Exercise 1.** Which method names does unittest run as tests? Pick them from the list.

In [ ]:
names = ["test_total", "check_total", "testEmpty", "helper_test"]
run = ___   # the names that start with "test"
check(1, run)

**Exercise 2.** Run this suite and count the failures.

In [ ]:
import unittest

class T(unittest.TestCase):
    def test_a(self):
        self.assertEqual(1 + 1, 2)
    def test_b(self):
        self.assertEqual("a".upper(), "a")

result = unittest.TextTestRunner(verbosity=0).run(unittest.defaultTestLoader.loadTestsFromTestCase(T))
predicted = ___   # len(result.failures)
check(2, predicted)

**Exercise 3.** Name the assertion that checks an error is raised: `self.assert___(ValueError)`.

In [ ]:
method = "assert" + ___
check(3, method)

## 2. setUp and isolation

- `setUp(self)` runs before *every* test, so each test gets fresh objects.
- Tests must not depend on each other or on run order.
- `self.subTest(value=v)` checks many cases in one test and reports each failure separately.

In [ ]:
import unittest

class CartTests(unittest.TestCase):
    def setUp(self):
        self.cart = []
    def test_add(self):
        self.cart.append("pen")
        self.assertEqual(len(self.cart), 1)
    def test_starts_empty(self):
        self.assertEqual(self.cart, [])
    def test_squares(self):
        for n, expected in [(2, 4), (3, 9)]:
            with self.subTest(n=n):
                self.assertEqual(n * n, expected)

r = unittest.TextTestRunner(verbosity=0).run(unittest.defaultTestLoader.loadTestsFromTestCase(CartTests))
print(r.wasSuccessful())

**Exercise 4.** Return the first score at or above the cutoff, in input order, or None when nobody qualifies. Repair `first_passing` and write `regression(candidate)`. Your test must pass for the repair and raise AssertionError for the original defect.

Earlier practice, for discussion:

A class has 3 test methods. How many times does `setUp` run?

```python
predicted = ___
```

In [ ]:
def first_passing(scores, cutoff):
    return next((n for n in scores if n > cutoff), None)

def regression(candidate):
    raise NotImplementedError


def observe(repair):
    def original(scores, cutoff):
        return next((n for n in scores if n > cutoff), None)

    regression(repair)
    try:
        regression(original)
    except AssertionError:
        detected = True
    else:
        detected = False
    inputs = [([5, 7, 6], 5), ([2, 3], 5), ([8, 6], 5)]
    return [repair(*args) for args in inputs], detected

check(4, first_passing, observe=observe)

**Exercise 5.** Count `setUp` calls for real.

In [ ]:
import unittest
calls = []

class T(unittest.TestCase):
    def setUp(self):
        calls.append(1)
    def test_one(self): pass
    def test_two(self): pass

unittest.TextTestRunner(verbosity=0).run(unittest.defaultTestLoader.loadTestsFromTestCase(T))
predicted = ___   # len(calls)
check(5, predicted)

## 3. Mocks

- `unittest.mock.Mock()` stands in for a collaborator (an API, a clock, a database). Set `mock.return_value` or `mock.side_effect`.
- After the call, check how it was used: `mock.assert_called_once_with(...)`, `mock.call_count`.
- Inject dependencies as parameters so tests can pass a mock instead of the real thing.

In [ ]:
from unittest.mock import Mock

def greet(user_id, fetch_name):
    return f"Hello, {fetch_name(user_id)}!"

fetch = Mock(return_value="Ana")
print(greet(7, fetch))
fetch.assert_called_once_with(7)

failing = Mock(side_effect=TimeoutError("slow"))
try:
    greet(1, failing)
except TimeoutError as error:
    print("raised:", error)

**Exercise 6.** Make the mock return `42`.

In [ ]:
from unittest.mock import Mock
get_price = Mock(return_value=___)
check(6, get_price("sku"))

**Exercise 7.** Call a mock three times. What is `call_count`?

In [ ]:
from unittest.mock import Mock
m = Mock()
m(); m("a"); m(x=1)
predicted = ___
check(7, predicted)

**Exercise 8.** Make the mock raise `ConnectionError` and catch it.

In [ ]:
from unittest.mock import Mock
fetch = Mock(side_effect=___)
try:
    fetch()
    outcome = "ok"
except ConnectionError:
    outcome = "failed"
check(8, outcome)

**Exercise 9.** Inject a fake clock so the function is testable.

In [ ]:
def greeting(now):
    return "morning" if now() < 12 else "afternoon"

results = (greeting(lambda: 9), greeting(lambda: ___))
check(9, results)   # make the second one "afternoon"

### Write an integration test with a fake clock

A fee service receives a clock and a repository. Charges submitted on the deadline are accepted. Add a fake repository, test the boundary and an expired request, and prove that rejected requests write nothing. Show that moving the deadline comparison by one instant breaks the test.

Read the ticket here and identify a failure case. After the independent build, use the engineering submission cell to implement it and run your regression tests. This ticket is reviewed from the submitted code and tests.

## Independent build

Write test_fee(candidate) for an order service. candidate(units, fee) returns fee for any positive unit count, otherwise zero. Inputs are nonnegative integers. Your test must pass for a conforming service and raise AssertionError for one that bills empty orders or charges per unit. Use fixed inputs.

In [ ]:
# Write your implementation here.

In [ ]:
def acceptance_cases(submission):
    def verify_candidate(candidate, should_pass):
        try:
            submission(candidate)
        except AssertionError:
            assert not should_pass, 'A conforming service failed your test'
        else:
            assert should_pass, 'A faulty service passed your test'
    verify_candidate(lambda n, f: f if n else 0, True)
    verify_candidate(lambda n, f: f, False)
    verify_candidate(lambda n, f: n * f, False)

check.assess('test_fee', acceptance_cases)

### Engineering submission

Write an integration test with a fake clock. Include your code, test results and review.

In [ ]:
# Implement the ticket and run your tests here, or submit the project files.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Where does a unittest.TestCase prepare fresh data for each test?

   - **a)** __init__
   - **b)** setUp
   - **c)** tearDownClass

**2.** Why mock an HTTP client in unit tests?

   - **a)** To hide bugs
   - **b)** Tests become fast, offline and deterministic
   - **c)** Mocks are more accurate

**3.** Which assertion checks that an error is raised?

   - **a)** assertEqual
   - **b)** assertRaises
   - **c)** assertTrue

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/14', {1: '412e05a94435', 2: '277d34fca480', 3: 'fcaedd512825'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [15 — SQLite, SQL, and Transactions](15_sqlite_and_transactions.ipynb)